# 28 — Validating Test A on past elections (pre-registration addendum 2)

Runs the checks fixed in `report/preregistration_2026_general_addendum_2.md` (committed 27 Sept 2026 before any of this data was downloaded). Specification unchanged from Test A: gap = candidate A two-party share − candidate B two-party share (points); WLS of the gap on the presidential swing to the Republican (points), HC1; units with < 200 votes in the weighting race dropped. Weights: Senate votes for Senate-vs-governor pairs (as Test A), governor votes for the Michigan 2018 down-ballot pairs (as addendum 1).

- **V1** Michigan 2018, municipalities (1,523 matched 2008–2018; every year's totals equal certified results exactly): V1a Nessel−Whitmer, Benson−Whitmer, Nessel−Benson (all newcomers); V1b Stabenow (incumbent) − Whitmer.
- **V2** positive controls, counties: Brown−Cordray (OH 2018), Baldwin−Evers (WI 2018), Fetterman−Shapiro (PA 2022). Presidential swings from the MIT county file. WI and PA equal certified totals exactly; Ohio's Senate Democrat equals the certified total exactly and the Republican is within 0.2%, governor within 0.2% (candidates matched by name within the race, since name formats vary by county).
- **V3** placebo swing 2008→2012 for all.

Swings: primary = the presidential swing before the election (2012→16 for 2018; 2016→20 for 2022); secondary = 2012→16 for all; placebo = 2008→12. Predictions (addendum 2): V1a within ±0.1; V1b no prediction; V2a, V2b positive under 2012→16; V2c no prediction.

In [1]:
import numpy as np, pandas as pd, statsmodels.formula.api as smf, re, warnings
warnings.filterwarnings('ignore'); pd.set_option('display.width', 220)
RAW = '../data/raw/'
def two(d, r): return 100 * d / (d + r)
def fit(df, gap, swing, wcol, label, extra=None):
    x = df[[gap, swing, wcol]].replace([np.inf, -np.inf], np.nan).dropna()
    x = x[x[wcol] >= 200]
    r = smf.wls(f'{gap} ~ {swing}', data=x, weights=x[wcol]).fit(cov_type='HC1'); ci = r.conf_int().loc[swing]
    return {'check': label, 'swing': swing, 'N': len(x), 'slope': r.params[swing], 'lo': ci[0], 'hi': ci[1], 'p': r.pvalues[swing], **(extra or {})}
rows = []

## V1 — Michigan 2018 (municipalities)

In [2]:
m = pd.read_csv(RAW + 'michigan/mi_municipal_validation_2008_2018.csv')
for o in ['gov', 'sen', 'ag', 'sos']: m[f'{o}_sh'] = two(m[f'{o}_D_2018'], m[f'{o}_R_2018'])
for y in (2008, 2012, 2016): m[f'rep{y}'] = 100 - two(m[f'pres_D_{y}'], m[f'pres_R_{y}'])
m['swing_12_16'] = m.rep2016 - m.rep2012; m['swing_08_12'] = m.rep2012 - m.rep2008
m['gov_tot'] = m.gov_D_2018 + m.gov_R_2018; m['sen_tot'] = m.sen_D_2018 + m.sen_R_2018
arab = m.muni.isin(['dearborn city', 'dearborn heights city', 'hamtramck city'])
pairs = [('V1a', 'Nessel (AG) - Whitmer (Gov)', 'ag_sh', 'gov_sh', 'gov_tot'), ('V1a', 'Benson (SoS) - Whitmer (Gov)', 'sos_sh', 'gov_sh', 'gov_tot'),
         ('V1a', 'Nessel (AG) - Benson (SoS)', 'ag_sh', 'sos_sh', 'gov_tot'), ('V1b', 'Stabenow (Sen, incumbent) - Whitmer (Gov)', 'sen_sh', 'gov_sh', 'sen_tot')]
for chk, lab, a, b, w in pairs:
    m['gap'] = m[a] - m[b]
    for sw, role in [('swing_12_16', 'primary'), ('swing_08_12', 'placebo')]:
        for sub, df in [('all', m), ('excl. Dearborn/Dearborn Hts/Hamtramck', m[~arab])]:
            rows.append(fit(df, 'gap', sw, w, f'{chk} MI 2018 {lab}', {'role': role, 'sample': sub, 'mean_gap': np.average(df.gap.dropna(), weights=df.loc[df.gap.notna(), w])}))
print(len(m), 'municipalities')

1523 municipalities


## V2 — positive controls (counties)

In [3]:
cp = pd.read_csv(RAW + 'mit_election_lab/countypres_2000-2024.csv', usecols=['state_po', 'county_name', 'county_fips'], dtype=str).drop_duplicates()
cr = pd.read_csv(RAW + 'mit_election_lab/county_returns.csv')
cr['fips'] = cr.county_fips.astype('Int64').astype(str).str.zfill(5)
pres = cr.pivot_table(index='fips', columns='year', values=['dem_votes', 'rep_votes'])
norm = lambda s: re.sub(r'[^a-z]', '', str(s).lower().replace('saint', 'st').replace('county', ''))
def county_frame(state_po, path, sen_names, gov_names, geo):
    d = pd.read_csv(RAW + 'validation/' + path, dtype=str, low_memory=False)
    d['v'] = pd.to_numeric(d.votes.str.replace(',', ''), errors='coerce')
    out = {}
    for key, office, names in [('sen', ['U.S. Senate', 'Senate'], sen_names), ('gov', ['Governor'], gov_names)]:
        s = d[d.office.isin(office)]
        for p, nm in names.items():
            out[f'{key}_{p}'] = s[s.candidate.str.contains(nm, case=False, na=False)].groupby('county').v.sum()
    f = pd.DataFrame(out); f['key'] = f.index.map(norm)
    names = cp[cp.state_po == state_po].assign(key=lambda x: x.county_name.map(norm), fips=lambda x: x.county_fips.astype(float).astype(int).astype(str).str.zfill(5))
    f = f.merge(names[['key', 'fips']].drop_duplicates('key'), on='key', how='left')
    miss = f[f.fips.isna()]
    print(state_po, len(f), 'counties; unmatched:', miss.index.tolist())
    for y in (2008, 2012, 2016, 2020):
        f[f'rep{y}'] = f.fips.map(lambda z: 100 * pres.loc[z, ('rep_votes', y)] / (pres.loc[z, ('rep_votes', y)] + pres.loc[z, ('dem_votes', y)]) if z in pres.index else np.nan)
    f['sen_sh'] = two(f.sen_D, f.sen_R); f['gov_sh'] = two(f.gov_D, f.gov_R); f['gap'] = f.sen_sh - f.gov_sh; f['sen_tot'] = f.sen_D + f.sen_R
    f['swing_08_12'] = f.rep2012 - f.rep2008; f['swing_12_16'] = f.rep2016 - f.rep2012; f['swing_16_20'] = f.rep2020 - f.rep2016
    return f
cases = [('V2a', 'OH 2018 Brown (Sen) - Cordray (Gov)', county_frame('OH', '20181106__oh__general__precinct.csv', {'D': 'Sherrod Brown', 'R': 'Renacci'}, {'D': 'Cordray', 'R': 'DeWine'}, 'county'), 'swing_12_16'),
         ('V2b', 'WI 2018 Baldwin (Sen) - Evers (Gov)', county_frame('WI', '20181106__wi__general__ward.csv', {'D': 'Baldwin', 'R': 'Vukmir'}, {'D': 'Evers', 'R': 'Walker'}, 'county'), 'swing_12_16'),
         ('V2c', 'PA 2022 Fetterman (Sen) - Shapiro (Gov)', county_frame('PA', '20221108__pa__general__county.csv', {'D': 'Fetterman', 'R': 'Oz'}, {'D': 'Shapiro', 'R': 'Mastriano'}, 'county'), 'swing_16_20')]
for chk, lab, f, prim in cases:
    for sw, role in [(prim, 'primary'), ('swing_12_16', 'secondary'), ('swing_08_12', 'placebo')]:
        if role == 'secondary' and sw == prim: continue
        rows.append(fit(f, 'gap', sw, 'sen_tot', f'{chk} {lab}', {'role': role, 'sample': 'all', 'mean_gap': np.average(f.gap, weights=f.sen_tot)}))
    print(f'  {lab}: statewide Senate D {f.sen_D.sum():,.0f}  Gov D {f.gov_D.sum():,.0f}')

OH 88 counties; unmatched: []
WI 72 counties; unmatched: []
PA 67 counties; unmatched: []
  OH 2018 Brown (Sen) - Cordray (Gov): statewide Senate D 2,358,508  Gov D 2,070,046
  WI 2018 Baldwin (Sen) - Evers (Gov): statewide Senate D 1,472,914  Gov D 1,324,307
  PA 2022 Fetterman (Sen) - Shapiro (Gov): statewide Senate D 2,751,012  Gov D 3,031,137


## Results

In [4]:
res = pd.DataFrame(rows)[['check', 'role', 'sample', 'swing', 'N', 'mean_gap', 'slope', 'lo', 'hi', 'p']]
res['outside_pm_0.1'] = (res.lo > 0.1) | (res.hi < -0.1)
print(res.round(3).to_string(index=False))
res.to_csv('../outputs/tables/testA_validation_addendum2.csv', index=False)

                                                check      role                                sample       swing    N  mean_gap  slope     lo     hi     p  outside_pm_0.1
              V1a MI 2018 Nessel (AG) - Whitmer (Gov)   primary                                   all swing_12_16 1423    -3.531 -0.085 -0.113 -0.057 0.000           False
              V1a MI 2018 Nessel (AG) - Whitmer (Gov)   primary excl. Dearborn/Dearborn Hts/Hamtramck swing_12_16 1420    -3.556 -0.084 -0.112 -0.055 0.000           False
              V1a MI 2018 Nessel (AG) - Whitmer (Gov)   placebo                                   all swing_08_12 1423    -3.531 -0.300 -0.432 -0.169 0.000            True
              V1a MI 2018 Nessel (AG) - Whitmer (Gov)   placebo excl. Dearborn/Dearborn Hts/Hamtramck swing_08_12 1420    -3.556 -0.295 -0.435 -0.155 0.000            True
             V1a MI 2018 Benson (SoS) - Whitmer (Gov)   primary                                   all swing_12_16 1423    -0.395  0.000 -0.0